In [127]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [128]:
import pandas as pd
import re
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [129]:
train_data = pd.read_csv('/content/drive/MyDrive/samsum-train.csv')
val_data = pd.read_csv('/content/drive/MyDrive/samsum-validation.csv')

In [130]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [131]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14732 entries, 0 to 14731
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        14732 non-null  object
 1   dialogue  14731 non-null  object
 2   summary   14732 non-null  object
dtypes: object(3)
memory usage: 345.4+ KB


# Data Preprocessing

In [132]:
train_data = train_data.dropna()

In [133]:
train_data.shape

(14731, 3)

In [134]:
val_data.shape

(818, 3)

In [135]:
def clean_data(text):
    text = re.sub(r'r/n/',' ',text)
    text = re.sub(r'\s+',' ',text)
    text = re.sub(r'<.*?>',' ',text)
    text = text.strip().lower()
    return text

In [136]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

# Tokenization

In [137]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

In [138]:
def tokenize(data):
    inputs = tokenizer(data['dialogue'],padding='max_length',max_length=512,truncation=True)
    target = tokenizer(data['summary'],padding='max_length',max_length=150,truncation=True)
    inputs['labels'] = target['input_ids']

    return inputs

In [139]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [140]:
train_dataset[0]
# input_ids => tokenized dialogue data
# attention_mask => 1 for real data 0 for padded data
# labels => input_ids of tokenized summary data

{'input_ids': [183, 232, 9, 10, 3, 23, 13635, 5081, 5, 103, 25, 241, 128, 58, 3, 12488, 651, 10, 417, 55, 183, 232, 9, 10, 3, 23, 31, 195, 830, 25, 5721, 3, 10, 18, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

# Model Building

In [141]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [142]:
import torch

if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')
print('device: ', device)
model.to(device)

device:  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [143]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/model',
    num_train_epochs= 6,

    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy='epoch',
    save_strategy='epoch',

    warmup_steps=500
)

In [144]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [145]:
trainer.train(resume_from_checkpoint=True)

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss


TrainOutput(global_step=11052, training_loss=0.0, metrics={'train_runtime': 0.0015, 'train_samples_per_second': 59918822.263, 'train_steps_per_second': 7492394.991, 'total_flos': 1.1962320464904192e+16, 'train_loss': 0.0, 'epoch': 6.0})

In [ ]:
model.save_pretrained('/content/drive/MyDrive/summary_model')
tokenizer.save_pretrained('/content/drive/MyDrive/summary_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
model = T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/summary_model')
tokenizer = T5Tokenizer.from_pretrained('/content/drive/MyDrive/summary_model')

Testing

In [ ]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue)

  inputs = tokenizer(
      dialogue,
      padding='max_length',
      max_length=512,
      truncation=True,
      return_tensors = 'pt'
  )

  targets = model.generate(
      input_ids = inputs['input_ids'],
      attention_mask = inputs['attention_mask'],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  summary = tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summarize_dialogue(test_dialogue)